# Edge model benchmark — Google Colab

Runs the **same tool-calling, pt-BR translation and summarization probes** as the local installer repo. Choose Runtime → Change runtime type → **T4 GPU** where available. Colab free compute is not guaranteed. No personal documents are needed.

This is not a Raspberry Pi or Radeon 780M speed benchmark. All tools are simulated: no shell commands generated by models are executed.


In [ ]:
import os, pathlib, subprocess
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
print(subprocess.getoutput("free -h | head -2"))
repo = pathlib.Path("/content/edge-llm-autoinstall")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/luandro/edge-llm-autoinstall.git", str(repo)], check=True)
else:
    print("Using existing checkout; review git status before updating:", subprocess.getoutput(f"git -C {repo} status --short"))


## Choose a test size
Start with `small`, quick suite. `edge` adds 4B and Gemma E4B; only use if the runtime has enough memory and disk. Optional paid GPUs may differ.


In [ ]:
MODELS = "small"  # small, edge, or comma-separated names: qwen08,botpsy,mini_agent,qwen2b
SUITE = "quick"  # quick (11 cases) or full (40 cases)
REPEATS = 1  # raise to 3 for meaningful comparisons
args=["bash", "benchmark.sh", "--models", MODELS, "--suite", SUITE, "--runs", str(REPEATS), "--backend", "auto", "--yes", "--max-download-gb", "16"]
print("Running", " ".join(args))
subprocess.run(args, cwd=str(repo), check=True)


In [ ]:
import glob, pathlib
from IPython.display import Markdown, display
reports=sorted(glob.glob(str(repo / "reports" / "*")))
if reports:
    latest=pathlib.Path(reports[-1])
    display(Markdown((latest/"comparison.md").read_text()))
    print("Results:", latest)
    # Optionally download report files using Colab:
    # from google.colab import files; files.download(str(latest/"comparison.csv"))
